# Explore observational data from Barcelona using ICOS Python Libraries

This example uses a foundational library called `icoscp_core` which can be used to access time-series ICOS data that are <i>previewable</i> in the ICOS Data Portal. "Previewable" means that it is possible to visualize the data variables in the preview plot. Here we use `cities` module in the library to access (meta-)data from [ICOS Cities](https://citydata.icos-cp.eu/portal/).

General information on all ICOS Carbon Portal Python libraries can be found on our [help pages](https://icos-carbon-portal.github.io/pylib/). Documentation of the `icoscp_core` library, including information on running it locally, can also be found on [PyPI.org](https://pypi.org/project/icoscp_core/).


* Authors: Vanessa Monteiro (ICTA-UAB) and Yi Wang (ICOS Carbon Portal)
* For questions, comments, suggestions, you can contact: Vanessa.Monteiro@uab.cat / yi.wang@mgeo.lu.se

## Data source

* Mole fractions from in-situ stations in Barcelona, Spain: https://citydata.icos-cp.eu/portal/
* 'Working with weather data': publicly available data retrieved from the Barcelona–El Prat Airport weather station (accessed via [Iowa Environmental Mesonet](https://mesonet.agron.iastate.edu/))

## Disclaimer

*These preliminary analyses (i.e., interpreting the figures) are presented solely as an illustrative example, with formal, validated results to be made available in future scientific publications.*

## Interactive plots

Most of the plots are interactive, so take time to explore the tools in the top right corner and you can hover the mouse over the data points.

**Note:** you can run specific notebook cells without running the entire notebook. However, you must run at least the "Explore stations and datasets" section to load the data before proceeding to the analysis.

--------------------------

## Explore stations and datasets

### Import libraries

In [ ]:
from icoscp_core.cities import data as citydata, meta as citymeta
from icoscp_core.cities import auth
from scripts.obs_explorer import *
from scripts.weather_explorer import *
import pandas as pd

In [ ]:
# uncomment to get help
# help(citymeta)

### Explore stations in a given country
In this section we list all city stations in Spain and make a map based on information we can retrieve from the station metadata

In [ ]:
# all stations
stations = citymeta.list_stations()

# extract stations in Spain
es_stations = [s for s in stations if s.country_code == 'ES']

**Convert station list to a pandas data frame**

Note: first station in the list above is for the entire Barcelona domain, not an observational station

In [ ]:
df = pd.DataFrame(es_stations)
df

**Create stations map**

In [ ]:
stations_map = create_station_map(df)
stations_map

### Explore metadata from a station

- In this section, we explore a single dataset or data object (with a given URI/PID) and its related metadata.

- For a single station you can use URI of the dataset you want to retrieve. This can be found on the ICOS Cities Data Portal at https://citydata.icos-cp.eu/portal/ 

**Example for station 'Fabra'**

In [ ]:
station_uri = "https://citymeta.icos-cp.eu/resources/stations/AS_ES_BCN_Fabra"
station_meta = citymeta.get_station_meta(station_uri)

In [ ]:
# print all metadata (a bit messy)
station_meta

Tip: to see what attribute you can get from the metadata, type `station_meta.` and then press tab

In [ ]:
# get station country
station_meta.countryCode

In [ ]:
station_meta.specificInfo

In [ ]:
# get station location
station_meta.location

In [ ]:
# get timezone offset (from UTC)
station_meta.specificInfo.timeZoneOffset

In [ ]:
uri = 'https://citymeta.icos-cp.eu/objects/QTFZV81ZT3CivlXo2qeJMP3E'

In [ ]:
# get all metadata from this data object
dobj_meta = citymeta.get_dobj_meta(uri)

Tip: to see what attribute you can get from the metadata, type `dobj_meta.` and then press tab

In [ ]:
# get lastest version of this data object (in case uri provided above is outdated)
dobj_meta.latestVersion

In [ ]:
# get title
dobj_meta.references.title

In [ ]:
# get citation string
dobj_meta.references.citationString

In [ ]:
# get BibTex
print(dobj_meta.references.citationBibTex)

In [ ]:
# get start and stop timestamp
dobj_meta.specificInfo.acquisition.interval

In [ ]:
# print column names
[col.label for col in dobj_meta.specificInfo.columns]

In [ ]:
# get sampling height
dobj_meta.specificInfo.acquisition.samplingHeight

In [ ]:
# access data and store to a pandas data frame
dobj_cols = citydata.get_columns_as_arrays(dobj_meta)
dobj_data = pd.DataFrame(dobj_cols)

In [ ]:
# explore first five rows of the data frame
dobj_data.head()

### Explore multiple datasets
- Here we create the dictionary to explore and visualize data from a single or multiple stations

**Compile data from all stations**

In [ ]:
# A list for all observational stations
ES_OBS = [
    s for s in stations
    if s.country_code == 'ES' and 'Barcelona_City' not in s.uri
]
pd.DataFrame(ES_OBS)

In [ ]:
# List data objects from all stations
dobj_list = citymeta.list_data_objects(station=ES_OBS)
pd.DataFrame(dobj_list)

In [ ]:
station_data = {}
station_meta = {}

for dobj in dobj_list:
    uri = dobj.uri
    station = dobj.station_uri.split("BCN_", 1)[-1]

    dobj_meta = citymeta.get_dobj_meta(uri)
    dobj_cols = citydata.get_columns_as_arrays(dobj_meta)
    dobj_data = pd.DataFrame(dobj_cols)

    # QC
    dobj_data = dobj_data[dobj_data["Flag"] == "O"].copy()

    # Set TIMESTAMP as index
    dobj_data["TIMESTAMP"] = pd.to_datetime(dobj_data["TIMESTAMP"])
    dobj_data = dobj_data.set_index("TIMESTAMP")

    station_data[station] = dobj_data
    station_meta[station] = dobj_meta

**Example for 'ICTA' station (printing the first 5 rows of the dataframe)**

In [ ]:
station_data['ICTA'].head()

In [ ]:
# Uncomment next line to see the example of how the metadata will look like for each station
#station_meta['ICTA']

In [ ]:
# Making sure that the Flag filtering was applied (Flag = 'O' means 'good data'),
# and it is the only flag you want to keep for the analysis.
station_data['ICTA']['Flag'].unique()

------------------------------
## Data availability


- Here you can see a summary of the data availability for the stations.
- This takes into account only data in the 'co2' column, assuming an hourly time series.


### Why is this information is important?

- Understanding data availability is crucial for any time series analysis because incomplete data can distort results and lead to wrong conclusions.
- In the next steps (time series), you will be able to identify where the gaps are.

In [ ]:
# Summary of data available for each dataset:
summary_availability = summarize_availability(station_data, var_name="co2")
summary_availability

The columns represent the following:

- Available Hours: The total number of hours with valid completed measurements.
- Missing/Bad Data: The number of hours with no measurements (for any reason) or data flagged as invalid/bad.
- Expected Hours: The total number of hours elapsed between the measurement start and end dates.
- Coverage: The percentage of valid data collected relative to the expected hours.

-------------------------------------
## Time series

- Now we make an interactive plot where it is possible to zoom in/out, pan, and hover over

### Why are these plots important?

- Here you can identify gaps and seasonal patterns, if present.
- It is also a way to assess data quality. A dataset with high data availability provides a reliable and consistent set of observations, giving you more confidence in your findings.
- This visualization is also helpful to guide data cleaning. Knowing where data is missing helps you decide whether to fill gaps (e.g., via imputation), trim invalid periods, or collect more data.
- This visualization easily helps ensure valid comparisons (e.g., comparing two locations or stations is only accurate if both have sufficient data over the same timeframe).

- Within the next sections you'll see a few examples of time series usage that can help you to interpret your CO$_2$ (carbon dioxide) and CH$_4$ (methane) datasets.

**Plotting options for Time Series:**

* *station_data_dict* : dictionary with data dataframes (e.g., 'station_data' in this notebook)
* *site_names* : list of site names that you want to display (e.g., ["ICTA","ICM"], default = None)
* *variable* : list of variables that you want to plot (e.g., ["co2","ch4"], default = "co2")
* *plot_type* : for time series use 'timeseries' (default= 'timeseries', options: "timeseries","diurnal", "excess","rose","excess_rose", "excessrose", "delta_rose")
* *resample_freq* 
* *split_day_night* : Options True or False 

### Single time series plot

In [ ]:
# Make the plot with a given time range
# Note: You can click on the legend to show/hide a station
# Standard single-station time series plot
plot_station_data_advanced(
    station_data_dict=station_data,
    site_names="ICTA",
    variable="co2",
    plot_type="timeseries",
    resample_freq="1h",
    split_day_night=False,
)

### Multiple time series plot

* Make the plot with a given time range
* Note: You can click on the legend to show/hide a station
* Standard multi-station time series plot

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data,
    site_names=["ICTA","ICM","IDAEA","Fabra","UPC"],
    variable="co2",
    plot_type="timeseries",
    start_date="2021-01-01",
    end_date="2026-01-01",
    split_day_night=False,
)

#### Interpreting the figures


* Looking at the time series we can detect a few characteristics of the mole fractions from the different stations, such as the seasonality. Across all stations, CO$_2$ values peak during late autumn and winter (December–February, reaching values >500 ppm at very urbanized stations) and drop during summer (July–August, near 415-430 ppm). This seasonal cycle is driven by vegetation uptake photosynthesis during summer and high planetary boundary layer (PBL), while during the winter, there is an increased combustion emission (residential heating) combined with lower PBL heights, that trap pollutants near the surface.

* Fabra exhibits a lower baseline and localized lower bounds, due to its positioning (forest), but still captures significant urban plume peaks during winter stagnation periods.

* ICM and IDAEA show frequent sharp local mole fractions and high-amplitude spikes, reflecting strong proximity to local traffic corridors, marine microclimates, or coastal anthropogenic sources.

* ICTA exhibits the highest mole fractions due to its location by the busiest highway of the region. 

* There is also a slight upward trend, reflecting global background increases superimposed on regional emissions.

### Linked plot between CO<sub>2</sub>, CH<sub>4</sub>, and any other gases/variables in the time series

- Here we create subplots that are linked on the x axis.
- When you zoom/pan in one subplot, the other one will also change accordingly. Double click to reset the scales.

#### Why are these plots important?

- Other than checking for data availability and trends, plotting multiple gases can help, for example, with source attribution. In the case of looking at CO$_2$ mole fractions and CH$_4$ mole fractions simultaneously makes it easy to spot co-located emission events. Synchronized spikes in both gases point to fossil fuel combustion or urban plumes, whereas isolated CH$_4$ spikes without elevated CO$_2$ suggest agricultural, sewage, or natural gas leak sources.

**Example of multiple gases time series on stacked subplots**

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data,
    site_names=["ICTA", "ICM", "Fabra"],
    variable=["co2", "ch4"],
    plot_type="timeseries",
    start_date="2023-06-01",
    end_date="2023-06-15"
)

**Example of multiple variables (in this case, standard deviation within the hour) time series on stacked subplots**

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data,
    site_names=["ICTA", "ICM", "Fabra"],
    variable=["co2", "co2_std","ch4", "ch4_std"],
    plot_type="timeseries",
    start_date="2023-06-01",
    end_date="2023-06-15"
)

#### Interpreting the figures

* All the plots above were created for a specific timeframe for demonstration.

Looking at them, we can infer a few things. 

**For the CO$_2$, CH$_4$ plots:**
* There is a strong diurnal cycle for both CO$_2$ and CH$_4$, with a pronounced daily oscillation. There are sharp peaks during late night/early morning hours, and the mole fractions drop to a baseline ($\sim$ 420 ppm for CO$_2$ and $\sim$ 2.0 ppm for CH$_4$) during mid-afternoon. This pattern is driven by nighttime atmospheric boundary layer shallowing (trapping urban emissions near the ground) combined with rush-hour traffic, followed by strong daytime convective mixing and vegetation uptake.
* The daily peaks in CO$_2$ and CH$_4$ align almost perfectly across time. For example, on Jun 2, Jun 4, and Jun 8, simultaneous spikes occur in both subplots. This synchronization strongly points to shared anthropogenic emission pathways, such as fossil fuel combustion, urban transport, or regional plume transport over Barcelona.

**Now, looking at the CO$_2$, the CO$_2$_STD, CH$_4$, and CH$_4$_STD**
* First, note that the suffix 'STD' denotes the standard deviation within the hour. The instruments used to measure atmospheric CO$_2$ and CH$_4$ in Barcelona sample at high frequency (approximately every 4 seconds), while the datasets retrieved from the ICOS Cities Data Portal provide the hourly average of these observations. Consequently, the STD column represents the variation around each hourly mean.
* High hourly standard deviations generally reflect a station's close proximity to local, intermittent emission sources (such as nearby vehicle traffic, industrial stacks, or sudden local combustion events) that create rapid, transient concentration spikes within that hour.
* Conversely, lower standard deviations indicate well-mixed atmospheric conditions or background baseline air that is farther removed from immediate point sources, where concentrations remain steady throughout the hour.
* From the figures shown above, we can infer that the peaks in CO$_2$_STD align closely with the peaks in the mole fractions (CO$_2$). When concentration increases alongside high standard deviation (e.g., STD spiking above 5 - 10 ppm), it confirms that the elevated values are driven by un-mixed, immediate local emissions (such as passing vehicle traffic or nearby combustion) passing directly over the inlet during that hour.
* Low STD indicates a well-mixed air mass. For example, during midday drop-offs, CO$_2$_STD drops to near zero (0.5 - 1 ppm) across all stations. This confirms that low daytime mole fractions generally represent well-mixed air, where atmospheric turbulence has evenly dispersed pollutants, resulting in steady readings over the hour.

* Note that the station ICTA, which is the closest to the highway (most traffic-intense area), exhibits both the highest concentration peaks ($\sim$ 500 ppm) and the highest within-hour variability (STD reaching 10 - 12 ppm). This strong variance indicates close proximity to highly dynamic emission sources or stronger local trapping under nocturnal boundary layers.
* IDAEA, for example, shows moderate concentration peaks paired with lower STD spikes ($\sim$ 3 - 6 ppm), suggesting it receives partially mixed plumes or sits slightly further back from immediate point sources.
  
* One potential use of this statistic (the within-hour STD) is data filtering. If you identify that patterns such as those described above match observations from the city you are analyzing, high STD values can be filtered out to isolate regional background trends, or targeted specifically if you are analyzing local, short-term urban pollution events.

### Split day and night

- All the previous plots can have the option to split the data visualization between day and night.
- You can define the boundaries of "day". Here in the example, "day" is defined as hours in between 8am (day_start=8) and 5pm (day_end=17), both inclusive. 

#### Why would you use these plots?

Splitting the time series in day vs night observation is useful in several aspects when screening your dataset. You can spot if the peaks on mole fractions are due to the atmospheric boundary layer dynamics, to vegetation activity, or really attributed to anthropogenic activities. 

* During the nighttime the planetary boundary layer shrinks into a shallow, stable layer near the ground. Because atmospheric mixing is minimal, surface emissions (from traffic, heating, etc.) get trapped, creating artificial concentration spikes that do not reflect total urban output.Also, there is no photosynthesis at night, but plants and soil continue to release CO$_2$ via cellular respiration (source), elevating background levels.
  
* During the daytime, the solar heating causes strong thermal convection, expanding the boundary layer (up to several kilometers high). This creates maximum atmospheric mixing, diluting local emissions, which reflects in minimum atmospheric concentrations. Also, the vegetation actively absorbs CO$_2$ through photosynthesis, creating a natural drawdown (sink) in the CO$_2$ mole fractions.

* Additionally, traffic, heating/cooling, and industrial operations follow strict daily schedules. Segmenting day versus night helps to isolates peak rush-hour emissions from  nighttime conditions.

**Example using two stations to plot CO2 split by day and night for a 15-day period**

Note: Here, we defined the day "start" and "end" hours. What is important here is being aware of the timeframe you want to distinguish in the graphs.

Select:

- day_start=8 (time when your 'day' starts)
- day_end=17 (time when your 'day' ends) 

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data,
    site_names=["ICTA","Fabra"],
    variable="co2",
    plot_type="timeseries",
    start_date="2025-01-01",
    end_date="2025-01-15",
    day_start=8,
    day_end=17,    
    split_day_night=True,
)

#### Interpreting the figures

* You can see that at ICTA, during the nighttime (hollow blue circles), there is a pronounced nighttime accumulation. There are much higher mole fractions and significant scatter (peaking between 460 and 525ppm). This behavior stems from nocturnal boundary layer, when there is low atmospheric mixing coupled with stable meteorological conditions, locking local combustion emissions near the surface overnight.

* During the daytime, ICTA (solid blue markers) mole fractions are noticeably lower and tighter, generally hovering between 425 and 440 ppm. Solar heating drives atmospheric convection, expanding the boundary layer and diluting local emissions into a well-mixed air mass.

* At FABRA station (orange markers), we note a less variable patter, with no high-amplitude spikes.This suggests that FABRA station is located at an elevated site, a coastal background location, or further away from dense urban traffic compared to ICTA. Because we know where FABRA is (a forested site, 415m ASL), we can confirm that the mole fractions at this station reflect the less polluted environment, and at night, the station is often above the boundary layer! 

-----------------------------------
## Annual growth rate

To calculate the annual growth rate, we use STL decomposition to extract the deseasonalized trend (the standard Python approach to approximate NOAA's curve fitting) and evaluate the trend values specifically at year-end (Dec 31) across a dictionary of site dataframes.

### Why look at the growth rates?

There are a few reasons to look at growth rates. Hourly-averaged mole fractions shift constantly with wind, weather, and planetary boundary layer changes. The annual growth rate removes these short-term dynamics to isolate whether local emissions (e.g., urban traffic, industrial sources, port activities) are actually accelerating or slowing down over time.

Thus, this can be useful to evaluate climate policy impact, for example. This metric can help tracking year-over-year changes to verify if low-emission zones, decarbonization policies, or local climate action plans are working. A declining growth rate indicates that emissions mitigation efforts are taking effect, while an increasing rate flags rising local activity.

The annual growth rates are also indicators of local sources. The global background CO$_2$ typically grows around 2 - 3 ppm/yr, thus strong local emission sources or regional atmospheric shifts will have greater growth rates. 

In [ ]:
# UNITS ARE ppm/year
growth_rate = calculate_noaa_growth_rate(station_data, col='co2', flag_col='flag', valid_flags=['O'])
display(growth_rate)

In [ ]:
growth_rate = calculate_noaa_growth_rate(station_data, col='ch4', flag_col='flag', valid_flags=['O'])
display(growth_rate)

### Interpreting the table

One way to check if a growth or drop is statistically significant (i.e., real change vs. noise), check if the range includes zero:
* Significant Growth:
  - If $Value - Uncertainty > 0$.
* Inconclusive / Statistically Zero:
  - If the uncertainty is larger than the value itself ($Value < Uncertainty$).

--
Examining the results for Barcelona:

* CO$_2$ (Top Table): We see a persistent positive growth across all stations from 2022 to 2025. IDAEA shows a sharp acceleration in growth rates starting in 2024 (7.19 and 7.77 ppm/yr), significantly higher than background atmospheric global averages (~2 - 3 ppm/yr).
  
* CH$_4$ (Bottom Table): We see transitions from negative/flat growth in 2022–2023 to strong positive growth by 2024–2025 across all sites.IDAEA experiences a massive spike in methane growth in 2025 (46.2 $\pm$ 10.0 ppb/yr), pointing to a strong local source emission event or major local shift. At ICM, in 2023 (-0.5 $\pm$ 8.7) ranges from -9.2 to +8.2. The uncertainty spans zero, meaning concentrations were effectively flat or the data was too noisy to confirm a change.

Further analysis are undergoing to identify the resons for the observed growth rate patterns at each site. 

----------------------------------------
## Diurnal cycle

### Why do we care about diurnal cycles?

Diurnal cycles reveal critical information about local greenhouse gas sources, atmospheric dynamics, and human behavior. 

Different activities peak at specific times of the day. A good example is the morning and evening peak in carbon dioxide (CO$_2$), which typically points to vehicular traffic during rush hours. 

Looking at the diurnal cycle we can also understand the gases behaviour as a function of the planetary boundary layer changes. At night, the planetary boundary layer shrinks and forms a shallow surface inversion, trapping gases near the ground and causing concentrations to rise. During the day, solar heating deepens the mixing layer, diluting surface emissions. Diurnal plots help separate changes in actual emissions from changes driven by boundary layer ventilation.

This is also an important visualization used for model validation. Atmospheric transport and chemistry models rely on accurate daily emission profiles. Comparing modeled diurnal curves against real station data is one of the standard ways to calibrate and validate atmospheric transport models. **Visit our Jupyter notebook comparing model outputs with observations!**


**Variability: what does it mean in these figures?**

* std (Standard Deviation)

Is the overall spread or dispersion of hourly data around the mean. 
A wide std band at a specific hour means that on any given day, mole fractions vary wildly (e.g., some days have huge traffic spikes while others are quiet). A narrow std band means the values at that hour are consistently predictable day after day.

* sem (Standard Error of the Mean - Default)

Is the precision of your calculated hourly average (Standard Deviation divided by $\sqrt{N}$).
It tells how confident we  are in the true average value for a given hour. It is best used when comparing two stations or two years to see if their mean diurnal curves are statistically distinct from one another.

* ci95 (95% Confidence Interval)

Is the range in which you can be 95% confident that the true population mean for that hour lies (roughly $\pm$ 1.96 $\times \text{SEM}$).
If the ci95 bands of two different monitoring sites (or two different seasons at the same site) do not overlap at a certain hour, you can conclude with 95% certainty that their average mole fractions at that hour are genuinely different.

* iqr (Interquartile Range: 25th to 75th percentile)

The middle 50% of your data, typically used alongside the median line.
It shows typical conditions while ignoring extreme outliers. Unlike std or sem, iqr is non-parametric, meaning it isn't skewed by sudden, massive pollution events (like a nearby fire or sensor glitch). If your data is skewed or dirty, iqr gives the most realistic view of standard daily behavior.

***In summary:***

Use std to see real day-to-day variability and extremes.

Use sem or ci95 to prove whether differences between sites, months, or years are statistically real.

Use iqr if your dataset has strong outliers or skewed distributions.




### Diurnal cycle by gases

Select:
- plot_type ="diurnal",
- variability = choose the variability type that you want displayed (default="sem", which stands for standard error, 'std' for 1 standard deviation, 'iqr' for interquartile, or  'ci95' for 95% confidence interval)

The following visualizations are examples of how the function for diurnal cycle can be used. 

**Example: CO2 diurnal cycle for all stations +- standard error**

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data,
    site_names=["ICTA","ICM",'IDAEA','UPC','Fabra'],
    variable="co2",
    plot_type="diurnal",
    variability= "sem",
)

**Example: CH4 diurnal cycle for all stations +- standard error**

In [ ]:
# Plot CH4 diurnal cycle for all stations
plot_station_data_advanced(
    station_data_dict=station_data,
    site_names=["ICTA","ICM",'IDAEA','UPC','Fabra'],
    variable="ch4",
    plot_type="diurnal",
    variability= "sem",
)

**Example: CO2 and CH4 diurnal cycle for two stations +- standard deviation**

In [ ]:
# diurnal cycle can also have multiple variables selected
plot_station_data_advanced(
    station_data_dict=station_data,
    site_names=["ICTA",'Fabra'],
    variable=["co2","ch4"],
    plot_type="diurnal",
    variability= "std",
)

#### Interpreting the figures

Note that these plots use the entire time series. At the time this notebook was created (August 2026), UPC has data roughly from January until May 2026, unlike the other stations that were running for a longer period of time. Keep in mind that comparing its results directly against the other stations would not be a fair comparison. For this reason, we will not focus on the interpretation of this specific site.

*The CO$_2$ diurnal cycle*

We can see that a prominent morning peak (05:00–07:00 UTC / 07:00–09:00 Local time) in the mole fractions. This aligns directly with Barcelona’s morning rush-hour traffic (e.g., heavily trafficked corridors like highways (AP-7), Diagonal, Gran Via, or the Ronda ring roads) and early-morning residential activity.

In the middle of the day (12:00–15:00 UTC), the mole fractions drop due to intense atmospheric mixing (deep PBL) and daylight photosynthetic uptake by urban vegetation and regional forests (e.g., Collserola natural park).
  
Even though evening rush hour and domestic activity in Barcelona generate comparable total emissions to the morning, the evening peak appears much smaller and broader on these mole fraction graphs. This asymmetry is caused by a combination of mechanisms. First, let's consider what happens to the planetary boundary layer (PBL). In the morrning the PBL is extremely shallow ($\sim$ 100 - 300 m) due to nocturnal cooling. Fresh emissions are squeezed into a tiny vertical volume, creating hyper-concentrated ground-level mole fractions.In the evening, the ground surface remains warm from daytime solar heating. The convective boundary layer stays deep ($\sim$ 1000 - 1500 m) well into late afternoon and early evening (17:00–20:00 UTC). Because of this, the evening emissions are diluted into a volume 5 to 10 times larger, thus we can't see a huge mole fractions peak in the evening. One other mechanism that happens in coastal areas, such as Barcelona, is that the sea breeze dynamic peaks during late afternoon and early evening. These onshore winds blow fresh, unpolluted marine air across the urban plain, flushing out accumulated pollutants and greenhouse gases right when evening traffic builds up. A last mechanism that can also explain the evening mole fractions is that morning commutes are highly compressed into a sharp 2-hour window as school and work start simultaneously. But, evening traffic, dining, and social activities are spread out over a much wider timeframe (e.g., from 18:00 to 22:00 Local Time), dispersing the emissions over time.



*The CH$_4$ diurnal cycle*

The role of the PBL over the day is the same for both gases, but sources of methane do not have a sharp spike at rush hours. Methane steadily builds up overnight and its emissions are associated to municipal natural gas distribution network leaks, sewer systems, and waste facilities/landfills (e.g., Zona Franca / Besòs areas). And, instead of the photossynthetic uptake in the mid-afternoon, methane suffers, photochemical oxidation ($\text{OH}$ radical chemistry).


*Site-by-site notes*

ICTA is located at the Universitat Autònoma de Barcelona in Cerdanyola del Vallès, close to one of the busiest highways in the region (AP-7). For this reason, its diurnal cycle shows the sharpest morning peak for CO$_2$.

ICM is a coastal site that experiences a higher marine boundary layer influence, resulting in a slightly dampened peak compared to ICTA.

Fabra is located on Mount Tibidabo at an elevation of $\sim$ 415 m. 
Because it sits above the nocturnal boundary layer inversion height most nights, its mole fractions are almost constant across the day, subject only to local traffic and incoming urban air masses that reach the site mainly during the day.

### Diurnal cycle by seasons

These are examples of how you can choose specific seasons.

**Note that** the function was created for Northern Hemisphere seasons, so it needs to be adjusted for other locations:

- summer = June, July, August
- autumn or fall = September, October, November
- winter = December, January, February
- spring = March, April, May


Select:

- seasons = 'summer', 'autumn' or 'fall', 'spring', 'winter'
- years = the default option includes all years from the dataset, but you can choose any specific year.

**Example of a 'summer diurnal cycle'**

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data,
    site_names=["ICTA","ICM",'IDAEA','Fabra'],
    variable="co2",
    plot_type="diurnal",
    seasons='summer',
    years=2024,
    variability= "sem",
)

**Example of a 'winter diurnal cycle'**

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data,
    site_names=["ICTA","ICM",'IDAEA','Fabra'],
    variable="co2",
    plot_type="diurnal",
    seasons='winter',
    years=2024,
    variability= "sem",
)

#### Interpreting the figures

Looking at these two diurnal cycles, we can see how patterns change during the summer (June, July, August) and the winter (December, January, February). Once again, there are a few mechanisms that affect these, which were already mentioned in the previous explanations, such as the chances in the atmospheric dynamics (boundary-layer mechanics), the photosynthetic activity, and local combustion.

* One of the most notable differences is in the mole fractions. While daytime minimum values converge near 422 - 425 ppm in the summer, these values are significantly higher in the winter, near 435 - 440 ppm. In the Northern Hemisphere, summer vegetation actively draws down global and regional CO$_2$, whereas plant dormancy and increased winter fossil fuel/heating emissions raise the overall background baseline. Not only that, but the depth of the planetary boundary layer also differs between seasons. During the summer, the PBL is deeper than in the winter, causing an effect similar to the 'day vs. night' changes explained before, where a shallow PBL leads to higher mole fractions.

* There is also a morning peak timing shift. You can see that during the summer the mole fractions peaks early around 05:00 UTC (07:00 Local Time), while in the winter, it peaks 2 hours later at 07:00 UTC (08:00 Local Time). Changes in human habits between the seasons added to an early vs late sunrise can help to explain these patterns, since early summer sunrise triggers thermal convection early in the morning, breaking up the shallow nocturnal boundary layer quickly, and a later sunrise delays solar heating, trapping morning rush-hour emissions under a persistent, shallow inversion layer for much longer. In the winter human commute schedules (schools, office openings around 08:00–09:00 local / 07:00–08:00 UTC) coincide with the period when the boundary layer is still shallow and trapped under a nocturnal inversion. However, in the summer, specifically July and August, schools close, and many business activities operate under alternative schedules or close for annual August vacations, which makes emissions to be either reduced or redistributed during the morning commuting traffic. Other factor that might play a role during the summer is that to beat the summer afternoon heat, construction work, outdoor municipal operations, logistics/deliveries, and agricultural labor start earlier in the morning (often by 06:00–07:00 local time), pushing early-morning emissions forward, alongside with the increase in tourism and nighttime hospitality activity, spreading traffic late into the night and shifting early-morning movements.

* We can also note the differences in the evening peak amplitude. The summer evening peak (18:00–22:00 UTC) is extremely weak (mole fractions barely reach 430 - 435 ppm), because the factors explained just above combined with the long daylight hours maintain a deep convective boundary layer, and strong coastal sea breezes continuously flush out urban emissions. While in the winter, it is noticeble a strong, sharp evening peak forms across all urban stations ($\sim$ 455 - 465 ppm), because the rapid cooling after early sunset collapses the boundary layer early, trapping evening traffic and domestic gas heating emissions near the ground.


* An interesting behaviour observed at Fabra is that during the summer, there is a daytime "bump" (peaking $\sim$ 07:00 - 08:00 UTC). Intense summer solar heating causes the planetary boundary layer to expand past $\sim$ 415 m, pushing urban air masses up Mount Tibidabo to the observatory. However, during the winter,  the mole fractions are relatively 'flat' all day ($\sim$ 432 - 435 ppm), because the shallow winter inversions keep urban pollution trapped in the Barcelona plain below, which leaves Fabra most of the time isolated in cleaner, free-tropospheric air above.

### Diurnal cycle by specific time range

* Sometimes, we might be interested in analyzing a specific timeframe. Below are a few examples showing how to select specific dates and compare them to the overall time series, uncovering the nuances of different dataset splits.

Select:

- start_date =  "yyyy-mm-dd"
- end_date = "yyyy-mm-dd"

Where: **yyyy** is the year, **mm** is the month (e.g., 01, 02,...12), and **dd** is the day (e.g., 01, 02,...,31)

**Example: April 2023**

In [ ]:
# Diurnal cycle for specific timeframe: April 2023
plot_station_data_advanced(
    station_data_dict=station_data,
    site_names=["ICTA","ICM",'IDAEA','UPC','Fabra'],
    variable=["co2","ch4"],
    plot_type="diurnal",
    start_date="2023-04-01", 
    end_date="2023-04-30",
    variability= "sem",
)

**Example: December 2023**

In [ ]:
# Diurnal cycle for specific timeframe: December 2023
plot_station_data_advanced(
    station_data_dict=station_data,
    site_names=["ICTA","ICM",'IDAEA','UPC','Fabra'],
    variable=["co2","ch4"],
    plot_type="diurnal",
    start_date="2023-12-01", 
    end_date="2023-12-31",
    variability= "sem",
)

#### Interpreting the figures

The interpretation of these figures is not as different from the ones of 'summer vs winter'. What we can highlight here is the difference in the standard error. If you recall, the 'SEM' is the *'precision of your calculated hourly average (Standard Deviation divided by $\sqrt{N}$). It tells how confident we  are in the true average value for a given hour. It is best used when comparing two stations or two years to see if their mean diurnal curves are statistically distinct from one another.'*


So, instead of repeating the text from the previous section that would be somehow equivalent on the overall interpretation, let's think what the shaded bands (the SEM) represents when comparing the 'long term' vs the 'short term' observations. 

* When shaded SEM bands between stations do not overlap, such as during morning rush hours or nocturnal inversions, the difference in surface mole fractions is statistically significant. This confirms that local emission proximity and site elevation dictate station behavior at those hours. But, when SEM bands converge and overlap, typically during peak afternoon convective mixing (12:00–15:00 UTC), the mole fractions are statistically indistinguishable, demonstrating full atmospheric homogenization across the urban region. In different words, if the SEM bands overlap, we cannot say that the stations have distinguished behavious. 

* Thus, averaging over multi-year periods filters out transient meteorological noise, isolating the structural climatology and site-specific baseline characteristics. Long-term trends highlight persistent spatial disparities between the sites, such as the 'background status' of Fabra Observatory relative to urban stations. They also reflect structural emission schedules like rush-hour traffic and annual vegetative drawdown. While short-term observations, for example single month (e.g., April vs. December 2023), reveals more episodic synoptic drivers and microclimatic shifts. 


* Overall, we can say that the high sample size shrinks the SEM bands into narrow envelopes, providing strong statistical power to declare even minor concentration offsets between sites as permanent, structural differences; and, the smaller sample size expands the SEM bands, capturing day-to-day weather volatility within that specific month (e.g., wider bands at specific hours, such as December evening peaks, show times when localized, acute meteorology heavily drives atmospheric variance).


### Diurnal cycle of standard deviation within the hour

#### Why do we also want to look at the standard deviation within the hour?

As explained in previous sections, the suffix STD (within these Jupyter notebooks) denotes the standard deviation within the hour. The instruments used to measure atmospheric CO$_2$ and CH$_4$ in Barcelona sample at high frequency (approximately every 4 seconds), while the datasets retrieved from the ICOS Cities Data Portal provide the hourly average of these observations. Consequently, the STD column represents the variation around each hourly mean.
* High hourly standard deviations generally reflect a station's close proximity to local, intermittent emission sources (such as nearby vehicle traffic, industrial stacks, or sudden local combustion events) that create rapid, transient concentration spikes within that hour.
* Conversely, lower standard deviations indicate well-mixed atmospheric conditions or background baseline air that is farther removed from immediate point sources, where concentrations remain steady throughout the hour.

***Here, it is important to distinguish between two types of standard deviation:***

*Within-hour standard deviation:* Measures the variability of high-frequency measurements taken inside a single hour.

*Standard deviation of the hourly mean:* Measures how much the average value for a specific hour (e.g., 8:00 AM) varies across different days.


**Coefficient of Variability (CV), i.e., concentration fluctuation intensity**

Sometimes, the high average of the nighttime STD within the hour is an artifact of the elevated mean concentration caused by a shallow nighttime boundary layer trapping emissions. To better analyse the data, we can calculate the coefficient of variability (CV): 

$\text{CV} = \frac{\sigma}{\mu} \times 100\%$ 

The CV normalizes variability relative to the mean, while standard deviation measures spread in absolute physical units.

You will see that in this example for Barcelona, the STD within the hour and the CV have similar pattern.

**Example of plotting the diurnal cycle of the 'standard deviation within the hour'**

Select:

- variable = "co2_std" or "ch4_std" or "co_std" (if available for a given station)

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data,
    site_names=["ICTA","ICM",'IDAEA','UPC','Fabra'],
    variable=["co2_std","ch4_std"],
    plot_type="diurnal",
    #start_date="2022-01-01", end_date="2024-12-31",
    #seasons='winter',
    variability= "sem",
)

**Example of plotting the diurnal cycle of the coefficient of variability (CV) or  concentration fluctuation intensity**

Select:

- variable = "co2" or "ch4" or "co" (if available)
- plot_type = "diurnal",
- variability = "cv_line"

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data,
    site_names=["ICTA","ICM",'IDAEA','Fabra'],
    variable=["co2"],
    plot_type="diurnal",
    variability="cv_line",  # Plots CV (%) as the main y-axis curve
)

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data,
    site_names=["ICTA","ICM",'IDAEA','UPC','Fabra'],
    variable=["co2", "ch4"],
    plot_type="diurnal",
    variability="cv_line",  # Plots CV (%) as the main y-axis curve
)

#### Interpreting the figures


As with the mole fraction plots, these variability profiles clearly highlight the morning and evening rush hours. But, because standard deviation almost always scales with the magnitude of the mole fraction, this means that these intra-hour STD spikes can be artifacts driven primarily by elevated background concentrations, as explained earlier. However, the CV has a similar pattern, which can lead us to the following interpretation.



ICTA and IDAEA exhibit the highest morning CV spikes for CO$_2$ (06:00–08:00 UTC) and elevated nocturnal CV for CH$_4$. The high relative variance indicates that these sensors are located close to fresh, unmixed urban emission plumes, such as traffic along major road corridors or localized gas infrastructure, where discrete concentration "spikes" pass over the sensor before the atmosphere can homogenize them. Because we know where the stations are located, we can confirm this is true!

UPC and ICM stations display lower morning peaks and flatter overall curves. UPC shows a delayed, localized spike around 15:00 UTC for CO$_2$, likely reflecting hyper-local campus or localized traffic activity rather than regional arterial traffic, but recall this station does not have a long time series (thus we cannot generalize results from UPC). ICM (coastal) benefits from marine air circulation and sea breezes, which dilute fresh emission plumes faster and smooth out localized variability compared to inland street-level sites.

Fabra has a nearly flat mole fraction hovering near 0.2% – 0.4% across all 24 hours for both gases, sustaining its 'status' as a background site. Its distance (and elevation) from ground-level urban sources means that emissions likely went thorough atmospheric mixing by the time the air mass reaches the station, eliminating short-term plume fluctuations.

### Diurnal cycle of CO2 excess (CO2xs)


- Note: Here, the CO$_2$ excess is calculated by subtracting the measurements at an urban station from those at a forested station. The forested station serves as a proxy background for this example; however, further analyses are underway to identify the most appropriate CO$_2$ background for Barcelona.

#### Why do we use CO2 excess (CO2xs or  $\Delta$CO2)?

This metric represents the concentration of carbon dioxide above a baseline, typically the  background level of a specific domain (in our case, the urban domain):

$$\text{CO}_2\text{xs} = \text{CO}_2\text{measured} - \text{CO}_2\text{background}$$

One strong reason for using CO$_2\text{xs}$ instead of the hourly CO$_2$ mole fractions is because it removes the broad regional noise, leaving only the signal generated by the domain emissions, which in urban context, is usually the anthropogenic component. 

In the case there are measurements of other gases (e.g., CH$_4$ or CO), we can calculate enhancement ratios ($\Delta\text{CO} / \Delta\text{CO}_2$), which can be used for source attribution. 

**Example of CO$_2$xs diurnal cycle using 'Fabra' as the background station**

Select:

- urban_stations= ["ICTA","ICM","IDAEA"] (select the urban stations of your interest)
- plot_type="excess"
- background_station="Fabra" (choose the background station)

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data,
    urban_stations=["ICTA","ICM",'IDAEA'],
    variable="co2",
    plot_type="excess",
    background_station='Fabra',
    variability= "sem",
)

**Example of CO$_2$xs diurnal cycle for April 2023 using 'Fabra' as the background station**

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data,
    urban_stations=["ICTA","ICM",'IDAEA','UPC'],
    variable="co2",
    plot_type="excess",
    start_date="2023-04-01", 
    end_date="2023-04-30",
    background_station='Fabra',
    #seasons='winter',
    variability= "sem",
)

**Example of CO$_2$xs diurnal cycle for December 2023 using 'Fabra' as the background station**

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data,
    urban_stations=["ICTA","ICM",'IDAEA'],
    variable="co2",
    plot_type="excess",
    start_date="2023-12-01", 
    end_date="2023-12-31",
    background_station='Fabra',
    #seasons='winter',
    variability= "sem",
)

#### Interpreting the figures

Calculating the CO$_2$xs relative to a reference site is more advantegeous than evaluating raw mole fractions alone. The hourly averaged mole fractions is a mixture of local anthropogenic activity with broad regional air mass dynamics and hemispheric seasonal baselines. By subtracting reference concentrations, CO$_2$xs isolates the net urban emissions generated within Barcelona, enabling a direct, baseline-independent comparison of the CO$_2$ generated in the urban area between contrasting seasons like December and April, exemplified above. 

When stations are at the same altitude and height, this direct subtraction would normalize for boundary layer dynamics, demonstrating periods such as spring middays when convective mixing temporarily flushes local additions and restores city air to background levels. 

However, it is important to note that while Fabra serves as a useful higher-elevation baseline for Barcelona, it is not a pristine, isolated background site and remains subject to periodic urban influence. At the moment,  ongoing efforts are focused on establishing a more robust, long-term regional background reference. 

### Diurnal cycle of CO2xs by weekday vs weekend

#### Why are we interested in different patterns such as weekdays vs weekends?

A good reason for comparing weekdays to weekends is because we can kind of isolate the anthropogenic contribution to the mole fractions. Since nature does not follow our calendars, but human behavior changes, we can identify, for example, the traffic contribution.  Road traffic typically experiences a sharp drop on weekends (especially morning rush hours). By comparing weekday and weekend CO$_2$ excess, you can separate vehicle emissions from baseline urban sources like residential heating or industrial operations.

Also, measuring the reductions in major sector emissions on weekends helps modelers validate urban emission inventories and assess how effective targeted traffic interventions (e.g., low-emission zones) might be.

**Example of diurnal cycle of CO2xs during the weekdays**

**Option to choose weekday vs weekends**

Select:

- day_type='weekday' or 'weekend'

Where **weekday** corresponds to Monday to Friday, and **weekend** to Saturday and Sunday.

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data,
    urban_stations =["ICTA","ICM",'IDAEA'],
    variable="co2",
    plot_type="excess",
    day_type="weekday",
    variability= "sem",
    years=2024
)

**Example of diurnal cycle of CO2xs during the weekend**

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data,
    urban_stations=["ICTA","ICM",'IDAEA'],
    variable="co2",
    plot_type="excess",
    day_type="weekend",
    variability= "sem",
    years=2024,
)

#### Interpreting the figures

During the weekdays, every station exhibits a pronounced morning rush-hour peak, reaching its absolute maximum at 06:00 UTC (this is averaged over the whole time series in this example). ICTA 
But, during the weekends, the morning rush-hour spike vanishes across all sites. 

ICTA’s morning maximum drops from 43 ppm down to $\sim$ 32 ppm (a $\sim$ 25% reduction) and flattens into a broad overnight broad plateau that peaks around 05:00 UTC rather than a sharp 06:00 UTC spike. ICTA also shows the strongest "weekend effect." Its high sensitivity to weekday vehicular emissions causes its morning excess to collapse drastically on weekends.

--------------------------------------------------------------------------
## Working with weather data

**Single weather station**

This is an example of retrieving weather data and merging it with station dataframes.

So far, we have worked exclusively with data from in-situ instruments. However, if meteorological data is available (e.g., wind speed and wind direction), it can be loaded and merged with existing dataframes for further filtering and analysis.

Below is an example using data retrieved from the Barcelona–El Prat Airport weather station (accessed via [Iowa Environmental Mesonet](https://mesonet.agron.iastate.edu/)). But, you always must select the weather station closest to your GHG measurement site.

In future updates we aim to include examples of integrating data from multiple weather stations.

### Load weather data and explore weather

**Load the weather data from the weather station closest to your GHG station**

The function we use in here requires that you know the weather station name (e.g., from the EL Prat-Barcelona Airport it is 'LEBL').

You can consult it here, in 'select network': https://mesonet.agron.iastate.edu/request/download.phtml

In [ ]:
df_lebl = fetch_met_data(
    station="LEBL",
    vars_to_fetch=["drct", "sknt"], #the variables are wind direction and wind speed in knots. This function returns a column with wind speed in meters per second
    start_year=2021,
    end_year=2026,
    resample_hourly=True #The data retrieved is not usually hourly averaged
)

In [ ]:
# Show the first five rows of the dataframe 
df_lebl.head()

**Wind rose of the dataset retrieved**

In [ ]:
plot_windrose_plotly(
    df_lebl,
    station_name="LEBL",
    #hours=range(12,18),
    #months=[6, 7, 8],
)

### Merge weather and dataframes

In [ ]:
# Merging data (Note that here is specifically to merge wind speed and direction with the stations data)
station_data_with_wind = add_wind_to_station_data(
    station_data=station_data,
    df_weather=df_lebl,
    tolerance_minutes=30,
    flag_col="Flag",  # Filters rows where Flag == 0
    valid_flag_value='O',
)

**Example of a dataframe output**

In [ ]:
station_data_with_wind['ICTA'].head()

### Time series and diurnal cycle of wind speed

Here we can explore the weather data the same way we previously explored the GHG observations.

***Note on wind speed and wind direction availability in the following plots:***

The wind speed displayed for a specific station (e.g., ICM) is merged directly with that station’s greenhouse gas measurement record. Because the mole fractions time series frequently contain data gaps, wind speed and direction values are only shown for date and time stamps where concurrent mole fractions observations exist. Consequently, the following plots do not represent a complete, uninterrupted wind speed record for the site.

The 'ICM' station will be used for the following examples, given it is the station that has the closest wind pattern as the weather station used here.

**Time series of wind speed (m/s)**

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data_with_wind,
    site_names=["ICM"], # THE WIND DATA DISPLAYED IS THE DATA MERGED WITH THE SPECIFIED STATION
    variable="wind_speed_ms",
    plot_type="timeseries",
)

**Diurnal cycle of wind speed (m/s)**

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data_with_wind,
    site_names=["ICM"],
    variable="wind_speed_ms",
    plot_type="diurnal",
    variability= "sem",
)

### Filtering concentrations based on wind data 

Note: A minimum wind speed threshold of 1.5 m/s (min_speed=1.5) was applied. While optional, filtering out calm or low-wind conditions is recommended to exclude stagnant air masses, which tend to cause artificially elevated mole fractions due to localized accumulation.

**Example of using wind direction data filtering: "N", "NNE", "NNW"**

Select:

* min_speed=1.5 (removes 'low wind speed. means that we will only look at the data that falls above this wind speed, in m/s. choose any threshold)
* sectors= "N", "NNE", "NNW", "ESE", "SE", "SSE", "S", "SSW", "SW", "WSW", "W", "WNW", "NW","E","NE","ENE"



In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data_with_wind,
    plot_type="diurnal",
    #background_station="Fabra",
    site_names=["ICM"],
    variable="co2",
    min_speed=1.5,
    sectors= ["N", "NNE", "NNW"], #, "ESE", "SE", "SSE", "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW",
    #seasons="summer",
    #day_type="weekday",
    variability="sem",  # Choose 'std', 'sem', 'iqr', 'ci95', or None
)

**Example of using wind direction data filtering: "SSE", "S", "SSW"**

In [ ]:
plot_station_data_advanced(
    station_data_dict=station_data_with_wind,
    plot_type="diurnal",
    #background_station="Fabra",
    site_names=["ICM"],
    variable="co2",
    min_speed=1.5,
    sectors= ["SSE", "S", "SSW"], #, "ESE", "SE", "SSE", "S", "SSW", "SW", "WSW", "W", "WNW", "NW", "NNW",
    #seasons="summer",
    #day_type="weekday",
    variability="sem",  # Choose 'std', 'sem', 'iqr', 'ci95', or None
)

#### Interpreting the figures

***Disclaimer:*** *These plots serve as an illustrative example of how weather station data can be used to filter observations and better understand in-situ greenhouse gas measurements. In complex urban environments like Barcelona, data from a single meteorological site (e.g., such as the station at Barcelona–El Prat Airport used in here) does not fully capture the microclimate and localized air mass circulation at individual measurement stations. While applying a single weather dataset is a valid methodology for areas with homogeneous terrain, a single station cannot account for complex coastal or urban micro-wind patterns. Consequently, the figures presented here are primarily to demonstrate the filtering options that can be potentially used rather than to provide a formal scientific interpretation of local atmospheric transport.*


**Example of how you could interpret such plots:**

Looking at 'ICM', we can see that opposing wind direction filters separates contrasting air mass source sectors. From the Northerly we can distinguish the inland/urban flow (N, NNE, NNW), given the winds originating from this wind sector bring air masses heavily influenced by urban surface emissions. These air masses have elevated overall CO$_2$ mole fractions (peaking above 457 ppm during the 07:00 UTC rush hour) and maintains a high baseline throughout the afternoon (444 - 458 ppm). But, when we look at the southerly sector, we see the influence of the coastal/marine flow (SSE, S, SSW). These southerly winds transport cleaner marine air toward the site, causing a drastic drop in baseline CO$_2$ across all hours. Even the morning rush-hour peak drops from $\sim$ 457 ppm down to $\sim$ 440 ppm, while afternoon levels stabilize around a minimum of $\sim$ 430 - 431 ppm.

---------------------------------------------------------------------------
# Comments and future improvements

In the future, we aim to expand this Jupyter notebook to include additional data analysis options, such as "pollution roses" for GHG mole fractions and CO$_2$ excess. 

Note that CO$_2$ background must be determined individually for each city, and we intend to provide a function that can ingest user-supplied .csv files and merge them with the relevant analysis.

Any feedback or contributions to improve these functions are welcome!

*AI use: Portions of this text were reviewed for grammar and style using Gemini.*

______________________________________
# References

**The text, the choice of visualizations, and the interpretation were based on previous work listed below.**

Christen, A. (2014). Atmospheric measurement techniques to quantify greenhouse gas emissions from cities. Urban Climate, 10, 241-260. https://doi.org/10.1016/j.uclim.2014.04.006

de Fatima Andrade, M., Marques, M. T. A., de Medeiros, J. A. G., Souto-Oliveira, C. E., Franco, M. A., Kumar, P., et al. (2026). METROCLIMA-MASP: pioneering urban greenhouse gas monitoring in South America’s largest megacity. City and environment interactions, 100326.

Estruch, C., Curcoll, R., Morguí, J.A., Segura-Barrero, R., Vidal, V., Badia, A., Ventura, S., Gilabert, J., & Villalba, G. (2024). Exploring how the heterogeneous urban landscape influences CO2 concentrations: The case study of the Metropolitan Area of Barcelona. Urban Forestry & Urban Greening, 99, 128438. https://doi.org/10.1016/j.ufug.2024.128438

Grange, S.K., Rubli, P., Fischer, A., Brunner, D., Hueglin, C. & Emmenegger, L. (2025). The ZiCOS-M CO2 sensor network: measurement performance and CO2 variability across Zurich. Atmospheric Chemistry and Physics, 25(5), 2781-2806. https://doi.org/10.5194/acp-25-2781-2025

Karion, A., Callahan, W., Stock, M., Prinzivalli, S., Verhulst, K.R., Kim, J., Salameh, P.K., Lopez-Coto, I., & Whetstone, J. (2020). Greenhouse gas observations from the Northeast Corridor tower network. Earth System Science Data, 12(1), pp.699-717. https://doi.org/10.5194/essd-12-699-2020

Kim, J., Karion, A., Loh, Z., Miles, N. & Xueref-Rémy, I. (2022). B.b. Tower and elevated point measurement data analysis in IG3IS Urban Greenhouse Gas Emission Observation and Monitoring Good Research Practice Guidelines. World Meteorological Organization. Last accessed: 18 May 2025. Available at: https://library.wmo.int/idurl/4/58055

McKain, K., Wofsy, S. C., Nehrkorn, T., Eluszkiewicz, J., Ehleringer, J. R., & Stephens, B. B. (2012). Assessment of ground-based atmospheric observations for verification of greenhouse gas emissions from an urban region. Proceedings of the National Academy of Sciences, 109(22), 8423-8428. https://doi.org/10.1073/pnas.1116645109

Miles, N. L., Richardson, S. J., Lauvaux, T., Davis, K. J., Balashov, N. V., Deng, A., Turnbull, J. C., Sweeney, C., Gurney, K. R., Patarasuk, R., Razlivanov, I., Cambaliza, M. O. L., & Shepson, P. B. (2017b). Quantification of urban atmospheric boundary layer greenhouse gas dry mole fraction enhancements in the dormant season: Results from the Indianapolis Flux Experiment (INFLUX). Elementa: Science of the Anthropocene, 5:27, 1-22. https://doi.org/10.1525/elementa.127

Mitchell, L. E., Lin, J. C., Bowling, D. R., Pataki, D. E., Strong, C., Schauer, A. J., Bares, R., Bush, S. E., Stephens, B. B., Mendoza, D., & Mallia, D. (2018). Long-term urban carbon dioxide observations reveal spatial and temporal dynamics related to urban characteristics and growth. Proceedings of the National Academy of Sciences, 115(12), 2912-2917. https://doi.org/10.1073/pnas.1702393115

Nishihashi, M., Mukai, H., Terao, Y., Hashimoto, S., Osonoi, Y., Boer, R., Ardiansyah, M., Budianto, B., Immanuel, G.S., Rakhman, A., & Nugroho, R. (2019). Greenhouse gases and air pollutants monitoring project around Jakarta megacity. In IOP Conference Series: Earth and Environmental Science (Vol. 303, No. 1, p. 012038). IOP Publishing. https://doi.org/10.1088/1755-1315/303/1/012038

Rice, A., & Bostrom, G. (2011). Measurements of carbon dioxide in an Oregon metropolitan region. Atmospheric Environment, 45(5), 1138-1144. https://doi.org/10.1016/j.atmosenv.2010.11.026

Turnbull, J. C., Curras, T., Gurney, K., Hilton, T. W., Mueller, K., Vogel, F. & Yao, B. (2025). IG3IS Urban Greenhouse Gas Emission Observation and Monitoring Good Research Practice Guidelines - WMO GAW IG3IS Report 2025. Geneva: World Meteorological Organisation. https://doi.org/10.59327/WMO/GAW/314.

Verhulst, K. R., Karion, A., Kim, J., Salameh, P. K., Keeling, R. F., Newman, S., Miller, J., Sloop, C., Pongetti, T., Rao, P. & Wong, C. (2017). Carbon dioxide and methane measurements from the Los Angeles Megacity Carbon Project–Part 1: calibration, urban enhancements, and uncertainty estimates. Atmospheric Chemistry and Physics, 17(13), 8313-8341. https://doi.org/10.5194/acp-17-8313-2017

Xueref-Rémy, I., Milne, M., Zoghbi, N., Lelandais, L., Riandet, A., Armengaud, A., Gille, G., Lanzi, L., Oppo, S., Bregonzio-Rozier, L., Blanc, P.-E., Yohia, C., Piazzola, J., & Delmotte, M. (2023). Analysis of atmospheric CO2 variability in the Marseille city area and the north-west Mediterranean basin at different time scales. Atmospheric environment: X, 17, 100208. https://doi.org/10.1016/j.aeaoa.2023.100208